In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('../..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc

plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})

In [ ]:
folder = "/home/liangmj/repo/LBM-Program/amrlbm/bin"

fileVec = [
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx/forcetorque/forceCylinder/forcetorque0000.txt",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_noGeoUpdate/forcetorque/forceCylinder/forcetorque0000.txt",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_larger_noGeoUpdate/forcetorque/forceCylinder/forcetorque0000.txt",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_40dx_larger_noGeoUpdate/forcetorque/forceCylinder/forcetorque0000.txt",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_wider_noGeoUpdate/forcetorque/forceCylinder/forcetorque0000.txt",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_wider_larger_noGeoUpdate/forcetorque/forceCylinder/forcetorque0000.txt",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_wider_L42_noGeoUpdate/forcetorque/forceCylinder/forcetorque0000.txt"


]

rho0 = 1.0
rhoPhy = 1.204
# startStep = 20000
startStep = 0

stepRg = [
    [100000, 200000],
    [100000, 200000],
    # [100000, 200000],
    [100000, 200000],
    [100000, 200000],

]

rVec = [
    # 16.0,
    # 16.0,
    # 16.0,
    # 20.0,
    16.0,
    16.0,
    # 16.0,
]

spanVec = [
    1,
    1,
    1,
    1,
    1,
]

AVec = [ r * 2 * span for r, span in zip(rVec, spanVec) ]

U0Vec = [
    0.01,
    0.01,
    0.01,
    0.01,
    0.01,
]

nuVec = [
    # 0.0016,
    # 0.0016,
    # 0.0016,
    # 0.002,
    0.0016,
    0.0016,
    # 0.0016,
]

colorVec = [
    # "#05A361",
    # "#D33737",
    # "#001AFF",
    # '#FF5733',
    "#119100",
    "#B700FF",
]

markerVec = [
    'o',
    '^',
    's',
    "D",
    '*',
]

linestyleVec = [
    "-",
    "--",
    ":",
    "-.",
    "-",
]

labelVec = [
    # "Present LBM IBB+MovBody D32 L10",
    # "Present LBM IBB D32 L10 Y10",
    # "Present LBM IBB D32 L25 Y10",
    # "Present LBM IBB D40 L25 Y10",
    # "Present LBM IBB D32 L10 Y16",
    "Present LBM IBB D32 L25 Y16",
    "Present LBM IBB D32 L42 Y16",
]

gamma = 1.0
dxVec = [1.0 / (2 * r) for r in rVec]
ucVec = [uc(dx, rhoPhy, gamma) for dx in dxVec]



In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def t_to_tStar(step, uRefLB, rLB):
    return uRefLB * step / rLB

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec


In [ ]:


stepVec = []
timeVec = []
forceVec = [[], [], []]
torqueVec = [[], [], []]
tStarVec = []
# refFxVec = []
for case in range(0, len(fileVec)):
    stepCol = read_col_probe(fileVec[case], 0, 2)
    fx = read_col_probe(fileVec[case], 2, 2)
    fy = read_col_probe(fileVec[case], 3, 2)
    fz = read_col_probe(fileVec[case], 4, 2)
    tx = read_col_probe(fileVec[case], 5, 2)
    ty = read_col_probe(fileVec[case], 6, 2)
    tz = read_col_probe(fileVec[case], 7, 2)
    
    idxStart = int(np.abs(stepCol - startStep).argmin())
    stepVec.append(stepCol[idxStart:])
    timeVec.append(stepCol[idxStart:] * ucVec[case].factor_time)
    forceVec[0].append(fx[idxStart:])
    forceVec[1].append(fy[idxStart:])
    forceVec[2].append(fz[idxStart:])
    torqueVec[0].append(tx[idxStart:])
    torqueVec[1].append(ty[idxStart:])
    torqueVec[2].append(tz[idxStart:])
    tStarVec.append(t_to_tStar(np.array(stepVec[-1]), U0Vec[case], rVec[case]))
    print(min(tStarVec[-1]), max(tStarVec[-1]))


In [ ]:

figF, axF = plt.subplots(3, 2, figsize=(16, 8), facecolor='w', edgecolor='w', sharex=True)
for case in range(0, len(fileVec)):
    for i in [0, 1, 2]:
        axF[i][0].plot(stepVec[case], forceVec[i][case], c=colorVec[case], lw=1.5, label=labelVec[case])
        axF[i][1].plot(stepVec[case], torqueVec[i][case], c=colorVec[case], lw=1.5, label=labelVec[case])

axF[0][0].set_ylabel('Force-x')
# axF[0][0].set_xlim(25000, None)
# axF[0][0].set_ylim(0.01, 0.02)
axF[1][0].set_ylabel('Force-y')
axF[2][0].set_ylabel('Force-z')
axF[2][0].set_xlabel('Step')

axF[0][1].set_ylabel('Torque-x')
axF[1][1].set_ylabel('Torque-y')
axF[2][1].set_ylabel('Torque-z')
axF[2][1].set_xlabel('Step')


figF.tight_layout()

In [ ]:
refCdFileVec = [
    "../ref/cd_chen.csv",
    "../ref/cd_shu.csv"
]

refClFileVec = [
    "../ref/cl_chen.csv",
    "../ref/cl_shu.csv"
]

refCdTStarVec = [read_csv_col(refCdFileVec[0], 0, 1), read_csv_col(refCdFileVec[1], 0, 1)]
refCdCdVec = [read_csv_col(refCdFileVec[0], 1, 1), read_csv_col(refCdFileVec[1], 1, 1)]
refClTStarVec = [read_csv_col(refClFileVec[0], 0, 1), read_csv_col(refClFileVec[1], 0, 1)]
refClClVec = [read_csv_col(refClFileVec[0], 1, 1), read_csv_col(refClFileVec[1], 1, 1)]

In [ ]:
figC, axC = plt.subplots(3, 2, figsize=(16, 8), facecolor='w', edgecolor='w', sharex=True)
# figSt, axSt = plt.subplots(1, 1, figsize=(5, 4), facecolor='w', edgecolor='w')
# figCtmp, axCtmp = plt.subplots(1, 1, figsize=(6, 6), facecolor='w', edgecolor='w')
# axCtmpY = axCtmp.twinx()

figCdTmp, axCdTmp = plt.subplots(1, 1, figsize=(6, 6), facecolor='w', edgecolor='w')
figClTmp, axClTmp = plt.subplots(1, 1, figsize=(6, 6), facecolor='w', edgecolor='w')

axCdTmp.plot(refCdTStarVec[0], refCdCdVec[0], c='k', linestyle='--', label='Chen at al., FD/pseudo-spectral')
axCdTmp.plot(refCdTStarVec[1], refCdCdVec[1], c='k', linestyle=':', label='Shu at al., D2Q7 TLLBM')

axClTmp.plot(refClTStarVec[0], refClClVec[0], c='k', linestyle='--', label='Chen at al., FD/pseudo-spectral')
axClTmp.plot(refClTStarVec[1], refClClVec[1], c='k', linestyle=':', label='Shu at al. D2Q7 TLLBM')

for case in range(0, len(fileVec)):
    fcVec = [[], [], []]
    tcVec = [[], [], []]
    idxStart = int(np.abs(np.array(stepVec[case]) - stepRg[case][0]).argmin())
    idxEnd = int(np.abs(np.array(stepVec[case]) - stepRg[case][1]).argmin())
    dynPressure = 0.5 * rho0 * U0Vec[case]**2
    for i in [0, 1, 2]:
        forceCoef = forceVec[i][case] / (dynPressure * AVec[case])
        torqueCoef = torqueVec[i][case] / (dynPressure * AVec[case] * rVec[case] * 2)
        # axC[i][0].plot(stepVec[case], forceCoef, c=colorVec[case], lw=1.5, label=labelVec[case])
        # axC[i][1].plot(stepVec[case], torqueCoef, c=colorVec[case], lw=1.5, label=labelVec[case])
        axC[i][0].plot(tStarVec[case], forceCoef, c=colorVec[case], lw=1.5, label=labelVec[case])
        axC[i][1].plot(tStarVec[case], torqueCoef, c=colorVec[case], lw=1.5, label=labelVec[case])
        fcVec[i] = forceCoef
        tcVec[i] = torqueCoef
    forceXCoef = forceVec[0][case] / (dynPressure * AVec[case])
    forceYCoef = forceVec[1][case] / (dynPressure * AVec[case])

    # axCtmp.plot(tStarVec[case], forceXCoef, c=colorVec[case], lw=1.5, linestyle='-')
    # # axCtmp.plot(timeVec[case], forceYCoef, c=colorVec[case], lw=1.5, linestyle=linestyleVec[case])
    # axCtmpY.plot(tStarVec[case], forceYCoef, c=colorVec[case], lw=1.5, linestyle='--')

    #! unsteady compare
    axCdTmp.plot(tStarVec[case], forceXCoef, c=colorVec[case], lw=1.5, linestyle='-', label=labelVec[case])
    axClTmp.plot(tStarVec[case], forceYCoef, c=colorVec[case], lw=1.5, linestyle='-', label=labelVec[case])

    # axCdTmp.plot(tStarVec[case], forceXCoef, marker=markerVec[case], fillstyle='none', ms=6, mew=1, markevery=100, c=colorVec[case], lw=0, label=labelVec[case])
    # axClTmp.plot(tStarVec[case], forceYCoef, marker=markerVec[case], fillstyle='none', ms=6, mew=1, markevery=100, c=colorVec[case], lw=0, label=labelVec[case])


    idxSt = int(np.abs(stepCol - startStep).argmin())
    dt = stepCol[1] - stepCol[0]
    dtSampling = dt
    freq = np.fft.fftfreq(len(stepVec[case]), d=dtSampling)
    positiveMask = freq > 0
    freq = freq[positiveMask]
    fyFreqDomain = np.fft.fft(forceVec[1][case])
    fyFreqDomain = np.abs(fyFreqDomain[positiveMask])
    stVec = freq * rVec[case] * 2 / U0Vec[case]
    st = stVec[np.argmax(fyFreqDomain)]
    # axSt.plot(stVec, fyFreqDomain, c=colorVec[case], lw=1.5, label=labelVec[case])

    print(f"Re={U0Vec[case]*rVec[case]*2/nuVec[case]}")
    print(f"Cy min = {min(fcVec[1][idxStart:idxEnd])}, Cy max = {max(fcVec[1][idxStart:idxEnd])}")
    print(f"St={st:.6f}, {freq[np.argmax(fyFreqDomain)]}, startStep={stepVec[case][0]}")
    print(f"Mean Cx={np.mean(fcVec[0][idxStart:idxEnd]):.6f}, Mean Cy={np.mean(fcVec[1][idxStart:idxEnd]):.6f}, Mean Cz={np.mean(fcVec[2][idxStart:idxEnd]):.6f}")
    print(f"Mean Cmx={np.mean(tcVec[0][idxStart:idxEnd]):.6f}, Mean Cmy={np.mean(tcVec[1][idxStart:idxEnd]):.6f}, Mean Cmz={np.mean(tcVec[2][idxStart:idxEnd]):.6f}")

# axSt.set_xlim(0, 0.5)
# axSt.set_ylim(0, 100)
# axSt.set_xlabel("St")
# axSt.set_ylabel(r"$\tilde{F_y}$")
# axSt.legend(loc='upper right', frameon=False, fontsize=13)

axC[0][0].set_ylabel(r'$C_x$')
axC[1][0].set_ylabel(r'$C_y$')
axC[2][0].set_ylabel(r'$C_z$')
# axC[2][0].set_xlabel('Step')
axC[2][0].set_xlabel(r'$t^*$')

axC[0][0].legend(loc='upper right', frameon=False, fontsize=13)
# axC[0][0].set_ylim(0.5, 3.5)
# axC[1][0].set_ylim(-3, 0)
# axC[0][0].set_xlim(20, 120)

#! Shu at al., TLLBM
axC[0][0].set_xlim(0, 45)
axC[0][0].set_ylim(0.0, 6.0)
axC[1][0].set_ylim(-15, 5)

axC[0][1].set_ylabel(r'$C_{mx}$')
axC[1][1].set_ylabel(r'$C_{my}$')
axC[2][1].set_ylabel(r'$C_{mz}$')
axC[2][1].set_xlabel('Step')
figC.tight_layout()

# axCtmp.set_xlabel(r"$t^*$")
# axCtmp.set_xlim(20, 120)
# axCtmp.set_ylim(0.0, 2.0)
# axCtmp.set_ylabel(r'$C_{D}$')
# axCtmpY.set_ylim(-12, 0)
# axCtmpY.set_ylabel(r'$C_{L}$')
# axCtmp.legend(loc='upper right', frameon=False, fontsize=13)

axCdTmp.set_xlim(0, 45)
axCdTmp.set_ylim(0, 6)
axClTmp.set_xlim(0, 45)
axClTmp.set_ylim(5, -6)

axCdTmp.set_xlabel(r"$t^*$")
axClTmp.set_xlabel(r"$t^*$")
axCdTmp.set_ylabel(r"$C_{D}$")
axClTmp.set_ylabel(r"$C_{L}$")

axCdTmp.legend(loc='upper right', frameon=False)
axClTmp.legend(loc='lower right', frameon=False)

axCdTmp.tick_params(width=2.0, axis='both', direction='in')
for spine in axCdTmp.spines.values():
    spine.set_linewidth(2.0)

axClTmp.tick_params(width=2.0, axis='both', direction='in')
for spine in axClTmp.spines.values():
    spine.set_linewidth(2.0)